In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.metrics import confusion_matrix
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 255)

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
data_path = "/kaggle/input/datasets/rehanliaqat17/imbd-dataset/IMDB_Dataset.csv"

In [3]:
df = pd.read_csv(data_path)
df.head()

,review,sentiment
0,"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of v...",positive
1,"A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen-...",positive
2,"I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hearted comedy. The plot is simplistic, but the dialogue is witty and the characters are likable (even the well b...",positive
3,"Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.<br...",negative
4,"Petter Mattei's ""Love in the Time of Money"" is a visually stunning film to watch. Mr. Mattei offers us a vivid portrait about human relations. This is a movie that seems to be telling us what money, power and success do to people in the different situ...",positive


In [4]:
df.shape

(50000, 2)

In [5]:
df = df.iloc[:10000]
df.head

<bound method NDFrame.head of                                                                                                                                                                                                                                                               review  \
0     One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of v...   
1     A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen-...   
2     I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hearted comedy. The plot is simpli

In [6]:
df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [7]:
df['sentiment'].value_counts()

sentiment
positive    5028
negative    4972
Name: count, dtype: int64

In [8]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(17)

In [11]:
df.drop_duplicates(inplace=True)

In [12]:
df.duplicated().sum()

np.int64(0)

**Basic Preprocessing**
* Remove tags-HTML
* Lower case
* Remove stopwords

In [13]:
import re
def remove_tags(raw_text):
    cleaned_text = re.sub(re.compile('<.*?>'),'',raw_text)
    return cleaned_text

In [14]:
df['review'] = df['review'].apply(remove_tags)
df.head()

,review,sentiment
0,"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.The first thing that struck me about Oz was its brutality and unflinching scenes of violence, whi...",positive
1,"A wonderful little production. The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. The actors are extremely well chosen- Michael Sheen not only ...",positive
2,"I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hearted comedy. The plot is simplistic, but the dialogue is witty and the characters are likable (even the well b...",positive
3,"Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.OK, first of al...",negative
4,"Petter Mattei's ""Love in the Time of Money"" is a visually stunning film to watch. Mr. Mattei offers us a vivid portrait about human relations. This is a movie that seems to be telling us what money, power and success do to people in the different situ...",positive


In [15]:
df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.I would say the main appeal of the show is due to the fact that it goes where other shows wo

In [16]:
df['review'] = df['review'].apply(lambda x:x.lower())
df['review'][0]

"one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word.it is called oz as that is the nickname given to the oswald maximum security state penitentary. it focuses mainly on emerald city, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. em city is home to many..aryans, muslims, gangstas, latinos, christians, italians, irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.i would say the main appeal of the show is due to the fact that it goes where other shows wo

In [17]:
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')
sw_list = stopwords.words('english')

df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:"".join(x))

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [18]:
df['review'][0]

"onereviewersmentionedwatching1ozepisodehooked.right,exactlyhappenedme.thefirstthingstruckozbrutalityunflinchingscenesviolence,setrightwordgo.trustme,showfaintheartedtimid.showpullspunchesregardsdrugs,sexviolence.hardcore,classicuseword.itcalledoznicknamegivenoswaldmaximumsecuritystatepenitentary.focusesmainlyemeraldcity,experimentalsectionprisoncellsglassfrontsfaceinwards,privacyhighagenda.emcityhomemany..aryans,muslims,gangstas,latinos,christians,italians,irishmore....soscuffles,deathstares,dodgydealingsshadyagreementsneverfaraway.iwouldsaymainappealshowduefactgoesshowsdare.forgetprettypicturespaintedmainstreamaudiences,forgetcharm,forgetromance...ozmessaround.firstepisodeeversawstrucknastysurreal,sayreadyit,watchedmore,developedtasteoz,gotaccustomedhighlevelsgraphicviolence.violence,injustice(crookedguardswho'llsoldnickel,inmateswho'llkillordergetawayit,wellmannered,middleclassinmatesturnedprisonbitchesduelackstreetskillsprisonexperience)watchingoz,maybecomecomfortableuncomfortablev

In [19]:
df.head()

,review,sentiment
0,"onereviewersmentionedwatching1ozepisodehooked.right,exactlyhappenedme.thefirstthingstruckozbrutalityunflinchingscenesviolence,setrightwordgo.trustme,showfaintheartedtimid.showpullspunchesregardsdrugs,sexviolence.hardcore,classicuseword.itcalledoznickn...",positive
1,"wonderfullittleproduction.filmingtechniqueunassuming-old-time-bbcfashiongivescomforting,sometimesdiscomforting,senserealismentirepiece.actorsextremelywellchosen-michaelsheen""hasgotpolari""voicespattoo!trulyseeseamlesseditingguidedreferenceswilliams'dia...",positive
2,"thoughtwonderfulwayspendtimehotsummerweekend,sittingairconditionedtheaterwatchinglight-heartedcomedy.plotsimplistic,dialoguewittycharacterslikable(evenwellbreadsuspectedserialkiller).maydisappointedrealizematchpoint2:riskaddiction,thoughtproofwoodyall...",positive
3,"basicallythere'sfamilylittleboy(jake)thinksthere'szombiecloset&parentsfightingtime.thismovieslowersoapopera...suddenly,jakedecidesbecomerambokillzombie.ok,firstgoingmakefilmmustdecidethrillerdrama!dramamoviewatchable.parentsdivorcing&arguinglikerealli...",negative
4,"pettermattei's""lovetimemoney""visuallystunningfilmwatch.mr.matteioffersusvividportraithumanrelations.movieseemstellingusmoney,powersuccesspeopledifferentsituationsencounter.variationarthurschnitzler'splaytheme,directortransfersactionpresenttimenewyorkd...",positive


In [20]:
X = df.iloc[:,0:1]
y = df['sentiment']

In [22]:
X.head()

,review
0,"onereviewersmentionedwatching1ozepisodehooked.right,exactlyhappenedme.thefirstthingstruckozbrutalityunflinchingscenesviolence,setrightwordgo.trustme,showfaintheartedtimid.showpullspunchesregardsdrugs,sexviolence.hardcore,classicuseword.itcalledoznickn..."
1,"wonderfullittleproduction.filmingtechniqueunassuming-old-time-bbcfashiongivescomforting,sometimesdiscomforting,senserealismentirepiece.actorsextremelywellchosen-michaelsheen""hasgotpolari""voicespattoo!trulyseeseamlesseditingguidedreferenceswilliams'dia..."
2,"thoughtwonderfulwayspendtimehotsummerweekend,sittingairconditionedtheaterwatchinglight-heartedcomedy.plotsimplistic,dialoguewittycharacterslikable(evenwellbreadsuspectedserialkiller).maydisappointedrealizematchpoint2:riskaddiction,thoughtproofwoodyall..."
3,"basicallythere'sfamilylittleboy(jake)thinksthere'szombiecloset&parentsfightingtime.thismovieslowersoapopera...suddenly,jakedecidesbecomerambokillzombie.ok,firstgoingmakefilmmustdecidethrillerdrama!dramamoviewatchable.parentsdivorcing&arguinglikerealli..."
4,"pettermattei's""lovetimemoney""visuallystunningfilmwatch.mr.matteioffersusvividportraithumanrelations.movieseemstellingusmoney,powersuccesspeopledifferentsituationsencounter.variationarthurschnitzler'splaytheme,directortransfersactionpresenttimenewyorkd..."


In [23]:
y

0       positive
1       positive
2       positive
3       negative
4       positive
5       positive
6       positive
7       negative
8       negative
9       positive
10      negative
11      negative
12      negative
13      negative
14      positive
15      negative
16      positive
17      negative
18      positive
19      negative
20      positive
21      negative
22      positive
23      negative
24      negative
25      positive
26      positive
27      negative
28      negative
29      positive
30      positive
31      positive
32      negative
33      positive
34      negative
35      negative
36      negative
37      negative
38      positive
39      negative
40      negative
41      positive
42      negative
43      negative
44      positive
45      positive
46      negative
47      negative
48      positive
49      negative
50      positive
51      positive
52      positive
53      positive
54      negative
55      negative
56      negative
57      negative
58      positi

In [24]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y = encoder.fit_transform(y)

In [25]:
y

array([1, 1, 1, ..., 0, 0, 1])

In [28]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [30]:
X_train.shape

(7986, 1)

In [31]:
X_test.shape

(1997, 1)

In [35]:
# Applying BoW
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

In [38]:
X_train_bow = cv.fit_transform(X_train['review']).toarray()
X_test_bow = cv.fit_transform(X_test['review']).toarray()

In [39]:
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [53]:
print(type(X_train))
print(type(y_train))

print("X_train shape:", X_train.shape if hasattr(X_train, 'shape') else len(X_train))
print("y_train shape:", y_train.shape if hasattr(y_train, 'shape') else len(y_train))

print("X_train_bow shape:", X_train_bow.shape)
print("y_train shape:", y_train.shape)

<class 'pandas.core.frame.DataFrame'>
<class 'numpy.ndarray'>
X_train shape: (7986, 1)
y_train shape: (7986,)
X_train_bow shape: (1, 1)
y_train shape: (7986,)


In [55]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# 1. Convert to 1D
X_train = X_train.squeeze()
X_test = X_test.squeeze()
y_train = y_train.squeeze()
y_test = y_test.squeeze()

# 2. BOW - same vectorizer
bow = CountVectorizer()

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

# 3. GaussianNB
gnb = GaussianNB()

gnb.fit(X_train_bow.toarray(), y_train)

# 4. Prediction
y_pred = gnb.predict(X_test_bow.toarray())

# 5. Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

# 6. Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# 7. Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.642463695543315

Confusion Matrix:
[[455 497]
 [217 828]]

Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.48      0.56       952
           1       0.62      0.79      0.70      1045

    accuracy                           0.64      1997
   macro avg       0.65      0.64      0.63      1997
weighted avg       0.65      0.64      0.63      1997



In [56]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Random Forest Classifier
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train
rf.fit(X_train_bow, y_train)

# Prediction
y_pred = rf.predict(X_test_bow)

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.6499749624436655

Confusion Matrix:
[[439 513]
 [186 859]]

Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.46      0.56       952
           1       0.63      0.82      0.71      1045

    accuracy                           0.65      1997
   macro avg       0.66      0.64      0.63      1997
weighted avg       0.66      0.65      0.64      1997



In [59]:
cv = CountVectorizer(max_features=3000)

X_train_bow = cv.fit_transform(X_train).toarray()
X_test_bow = cv.transform(X_test).toarray()

rf = RandomForestClassifier()
rf.fit(X_train_bow, y_train)
y_pred = rf.predict(X_test_bow)
accuracy_score(y_test,y_pred)

0.6359539308963446

In [60]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# N-gram Count Vectorizer
cv = CountVectorizer(
    max_features=3000,
    ngram_range=(1, 2)
)

# Transform data
X_train_bow = cv.fit_transform(X_train).toarray()
X_test_bow = cv.transform(X_test).toarray()

# Random Forest Classifier
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train
rf.fit(X_train_bow, y_train)

# Prediction
y_pred = rf.predict(X_test_bow)

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.6314471707561342

Confusion Matrix:
[[595 357]
 [379 666]]

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.62      0.62       952
           1       0.65      0.64      0.64      1045

    accuracy                           0.63      1997
   macro avg       0.63      0.63      0.63      1997
weighted avg       0.63      0.63      0.63      1997



In [61]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=3000)

# Transform data
X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

# Random Forest Classifier
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train
rf.fit(X_train_tfidf, y_train)

# Prediction
y_pred = rf.predict(X_test_tfidf)

# Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.6459689534301453

Confusion Matrix:
[[601 351]
 [356 689]]

Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.63      0.63       952
           1       0.66      0.66      0.66      1045

    accuracy                           0.65      1997
   macro avg       0.65      0.65      0.65      1997
weighted avg       0.65      0.65      0.65      1997

